# SAR Pipeline — Clean Analysis Notebook
## Paper: XAI-based DSS for Agricultural Skill Development (AIA Journal)

**Sections:**
1. Imports & Setup
2. Data Loading & Feature Definitions
3. Farmer Segmentation (K-Means, k=3)
4. Predictive Modelling (RF + Baselines)
5. SHAP Analysis (Global + Cluster-level)
6. Surrogate Decision Tree (Rule Extraction)
7. Supplementary: Cluster Profile Heatmap

> **Reproducible:** Run all cells top-to-bottom.  
> All figures are saved to `./figures/`

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import shap
import warnings
import os
warnings.filterwarnings('ignore')

from pyprojroot import here
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                              davies_bouldin_score)
from sklearn.tree import DecisionTreeClassifier, plot_tree

import os
OUT = 'figures'
os.makedirs(OUT, exist_ok=True)
print("✓ Imports complete")

## 2. Data Loading & Feature Definitions

In [ ]:
# หาตำแหน่งของโฟลเดอร์ที่ไฟล์ code นี้วางอยู่
base_path = here()
file_path = os.path.join(base_path, "datas", "SFProgramDataPanal.csv")
# file_path = os.path.join(base_path, "data", "SFProgramDataPanal.csv")

df = pd.read_csv(file_path)

df = df.replace('.', np.nan)

# ── Feature sets ──────────────────────────────────────────────────────────
CLUSTER_FEATS = [
    'age', 'edu', 'agri_long', 'irriga', 'loan',
    'Avg_ProdManage', 'Avg_InputManage', 'Avg_Tech',
    'Avg_Ana&Plan', 'Avg_Mkting', 'Avg_Network',
    'Ave_ProdRisk', 'Ave_InputRisk', 'Ave_MktRisk', 'Ave_FinRisk'
]

RF_FEATS = [
    'age', 'edu', 'agri_long', 'gender', 'irriga', 'loan', 'region',
    'Avg_ProdManage', 'Avg_InputManage', 'Avg_Tech', 'Avg_Ana&Plan',
    'Avg_Mkting', 'Avg_Network',
    'Ave_ProdRisk', 'Ave_InputRisk', 'Ave_MktRisk', 'Ave_FinRisk',
    'sf_participant'
]

SHAP_FEATS = [
    'age', 'edu', 'agri_long', 'irriga', 'loan',
    'Avg_ProdManage', 'Avg_Tech', 'Avg_Mkting',
    'Ave_MktRisk', 'Ave_FinRisk', 'Overview_Risk',
    'agri Org_mem', 'gov_support', 'All_Skill'
]

SHAP_RENAME = {
    'age': 'Age', 'edu': 'Education', 'agri_long': 'Agri Exp',
    'irriga': 'Irrigation', 'loan': 'Loan',
    'Avg_ProdManage': 'Avg_ProdManage', 'Avg_Tech': 'Avg_Tech',
    'Avg_Mkting': 'Avg_Mkting', 'Ave_MktRisk': 'Mkt Risk',
    'Ave_FinRisk': 'Fin Risk', 'Overview_Risk': 'Overall Risk',
    'agri Org_mem': 'Agri Org', 'gov_support': 'Gov Support',
    'All_Skill': 'All_Skill'
}

OUTCOME        = 'Ch_Skill'
CLUSTER_NAMES  = {0: 'Low-skill', 1: 'Moderate-skill', 2: 'High-skill'}
CLUSTER_COLORS = {0: '#4472C4', 1: '#FF8C00', 2: '#2ECC71'}

print(f"Shape: {df.shape}")
print(f"Outcome ({OUTCOME}): mean={df[OUTCOME].mean():.3f}, "
      f"std={df[OUTCOME].std():.3f}, n={df[OUTCOME].notna().sum()}")
df.head(3)

## 3. Farmer Segmentation (K-Means, k=3)

### 3a. Cluster Validity Indices (k=2..8)

In [ ]:
df_cl = df[CLUSTER_FEATS].apply(pd.to_numeric, errors='coerce').dropna().copy()
scaler = StandardScaler()
X_sc = scaler.fit_transform(df_cl)
print(f"Clustering n = {len(df_cl)}")

ks = range(2, 9)
results = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_sc)
    results.append({
        'k':          k,
        'inertia':    km.inertia_,
        'silhouette': silhouette_score(X_sc, lbl),
        'calinski':   calinski_harabasz_score(X_sc, lbl),
        'davies':     davies_bouldin_score(X_sc, lbl),
    })

cv_df = pd.DataFrame(results)
cv_df.set_index('k')

### 3b. Figure: 4-panel Cluster Validity (Fig. 2 in paper)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
panels = [
    ('inertia',    'Within-cluster Inertia', 'Elbow Method',      '#1565C0'),
    ('silhouette', 'Silhouette Score',        'Silhouette',        '#1976D2'),
    ('calinski',   'Calinski-Harabasz Score', 'Calinski-Harabasz','#0277BD'),
    ('davies',     'Davies-Bouldin Score',    'Davies-Bouldin',   '#00796B'),
]
for ax, (col, ylabel, title, color) in zip(axes, panels):
    ax.plot(cv_df['k'], cv_df[col], 'o-', color=color, lw=2, ms=6)
    ax.axvline(3, color='#E53935', ls='--', lw=1.5, alpha=0.8, label='k=3')
    for k, v in zip(cv_df['k'], cv_df[col]):
        ax.annotate(f'{v:.3f}', (k,v), textcoords='offset points',
                    xytext=(0,8), ha='center', fontsize=7.5)
    ax.set(title=title, xlabel='Number of Clusters (k)', ylabel=ylabel)
    ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.suptitle('Cluster Validity: Four Complementary Indices (k=2–8)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_elbow_silhouette.png', bbox_inches='tight',
            dpi=180, facecolor='white')
plt.show()

### 3c. Fit k=3, Label Clusters, Build Profile Table

In [ ]:
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cl['Cluster_raw'] = km3.fit_predict(X_sc)

df_full = df.loc[df_cl.index].apply(pd.to_numeric, errors='coerce').copy()
df_full['Cluster_raw'] = df_cl['Cluster_raw'].values

# Label 0=Low, 1=Moderate, 2=High by mean All_Skill
order = df_full.groupby('Cluster_raw')['All_Skill'].mean().sort_values()
label_map = {old: new for new, old in enumerate(order.index)}
df_full['Cluster'] = df_full['Cluster_raw'].map(label_map)
df_full['Cluster_name'] = df_full['Cluster'].map(CLUSTER_NAMES)

# Profile table
profile_cols = ['All_Skill', OUTCOME, 'age', 'edu', 'agri_long',
                'Avg_ProdManage', 'Avg_Tech', 'sf_participant']
profile = df_full.groupby('Cluster')[profile_cols].mean().round(3)
profile['n'] = df_full.groupby('Cluster').size()
profile.index = ['Low-skill', 'Moderate-skill', 'High-skill']
profile

### 3d. Figure: PCA Cluster Scatter (Fig. 3 in paper)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)
ev = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(7, 6))
for raw_lbl, mapped_lbl in label_map.items():
    mask = df_cl['Cluster_raw'] == raw_lbl
    ax.scatter(X_pca[mask,0], X_pca[mask,1],
               c=CLUSTER_COLORS[mapped_lbl],
               label=CLUSTER_NAMES[mapped_lbl],
               alpha=0.65, s=25, edgecolors='none')
ax.set(title='Farmer Clusters (k=3, Silhouette=0.185)',
       xlabel=f'PC1 ({ev[0]:.1%})', ylabel=f'PC2 ({ev[1]:.1%})')
ax.legend(framealpha=0.9, fontsize=10); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(f'{OUT}/fig2_clusters_new.png', bbox_inches='tight',
            dpi=180, facecolor='white')
plt.show()

## 4. Predictive Modelling (RF + Baselines)

In [ ]:
df_rf = df[RF_FEATS + [OUTCOME]].apply(pd.to_numeric, errors='coerce').dropna().copy()
X_rf  = df_rf[RF_FEATS]
y_rf  = df_rf[OUTCOME].values
print(f"RF dataset: n={len(df_rf)}")

rows = []
for name, model in [
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression',  Ridge(alpha=1.0)),
    ('Random Forest (tuned)', RandomForestRegressor(
        n_estimators=500, min_samples_leaf=10,
        max_features='sqrt', random_state=42, n_jobs=-1)),
]:
    r2   = cross_val_score(model, X_rf, y_rf, cv=5, scoring='r2')
    rmse = -cross_val_score(model, X_rf, y_rf, cv=5,
                             scoring='neg_root_mean_squared_error')
    mae  = -cross_val_score(model, X_rf, y_rf, cv=5,
                             scoring='neg_mean_absolute_error')
    rows.append({'Model': name, 'R²': r2.mean().round(3),
                 'RMSE': rmse.mean().round(3), 'MAE': mae.mean().round(3)})

pd.DataFrame(rows).set_index('Model')

> **Note on GridSearch (optional):**  
> Best params found: `n_estimators=500, min_samples_leaf=10, max_features='sqrt'`  
> Run the cell below only if you want to re-confirm (takes ~3 min)

In [ ]:
# OPTIONAL — uncomment to re-run GridSearch
# param_grid = {
#     'n_estimators':    [300, 500],
#     'max_depth':       [5, 8, None],
#     'min_samples_leaf':[5, 10, 20],
#     'max_features':    ['sqrt', 0.5],
# }
# gs = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
#                   param_grid, cv=5, scoring='r2', n_jobs=-1)
# gs.fit(X_rf, y_rf)
# print(gs.best_params_, gs.best_score_)

## 5. SHAP Analysis

### 5a. Compute SHAP Values

In [ ]:
RF_PARAMS = dict(n_estimators=500, min_samples_leaf=10,
                 max_features='sqrt', random_state=42, n_jobs=-1)

df_shap = df[SHAP_FEATS + [OUTCOME]].apply(
    pd.to_numeric, errors='coerce').dropna().copy()
X_shap  = df_shap[SHAP_FEATS].rename(columns=SHAP_RENAME)
y_shap  = df_shap[OUTCOME].values

rf_shap = RandomForestRegressor(**RF_PARAMS)
rf_shap.fit(X_shap, y_shap)

explainer = shap.TreeExplainer(rf_shap)
sv = explainer.shap_values(X_shap)           # shape: (n_samples, n_features)
mean_shap = pd.Series(np.abs(sv).mean(0), index=X_shap.columns)

print("Global SHAP importance (all features):")
print(mean_shap.sort_values(ascending=False).round(4).to_string())

### 5b. Figure: Global SHAP Bar (Fig. 4 in paper)

In [ ]:
mean_shap_sorted = mean_shap.sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(mean_shap_sorted.index, mean_shap_sorted.values,
        color='#2980b9', edgecolor='white', linewidth=0.5)
ax.set_xlabel('mean(|SHAP value|)', fontsize=10)
ax.set_title('SHAP Feature Importance → Ch_Skill', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig3_importance_new.png', bbox_inches='tight',
            dpi=180, facecolor='white')
plt.show()

### 5c. Figure: Global SHAP Beeswarm (Fig. 5 in paper)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_shap, plot_type='dot', max_display=10, show=False)
plt.title('SHAP Beeswarm: Determinants of Skill Change (Ch_Skill)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4_shap_new.png', bbox_inches='tight',
            dpi=180, facecolor='white')
plt.show()

### 5d. Attach Cluster Labels to SHAP Dataset

In [ ]:
df_shap['Cluster'] = df_full.loc[
    df_shap.index.intersection(df_full.index), 'Cluster']
df_shap = df_shap.dropna(subset=['Cluster'])
df_shap['Cluster'] = df_shap['Cluster'].astype(int)
print(df_shap['Cluster'].value_counts().sort_index())

### 5e. Figure: Cluster-level SHAP Beeswarm 3-panel (Fig. 6 in paper)

In [ ]:
panel_titles = [
    f'Cluster 1: Low-skill\n(n={(df_shap["Cluster"]==0).sum()})',
    f'Cluster 2: Moderate-skill\n(n={(df_shap["Cluster"]==1).sum()})',
    f'Cluster 3: High-skill\n(n={(df_shap["Cluster"]==2).sum()})',
]

fig = plt.figure(figsize=(17, 5))
for ci in range(3):
    ax = fig.add_subplot(1, 3, ci + 1)
    idx = df_shap[df_shap['Cluster'] == ci].index
    sv_sub = explainer.shap_values(X_shap.loc[idx])

    # Top-5 features for this cluster
    top5 = (pd.Series(np.abs(sv_sub).mean(0), index=X_shap.columns)
              .sort_values(ascending=False).head(5).index.tolist())
    col_idx = [list(X_shap.columns).index(f) for f in top5]

    plt.sca(ax)
    shap.summary_plot(sv_sub[:, col_idx],
                      X_shap.loc[idx, top5].values,
                      feature_names=top5,
                      plot_type='dot', show=False,
                      plot_size=None, max_display=5)
    ax.set_title(panel_titles[ci], fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(labelsize=9)

plt.suptitle(
    'Cluster-Stratified SHAP: Segment-Specific Drivers of Skill Improvement',
    fontsize=12, y=1.03)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4b_cluster_shap_new.png', bbox_inches='tight',
            dpi=180, facecolor='white')
plt.show()

## 6. Surrogate Decision Tree (Rule Extraction)

**Two-step rule extraction:**
1. **Step 1 — Feature selection:** SHAP selects top-K features by mean |SHAP|
2. **Step 2 — Threshold calibration:** Surrogate decision tree (depth=3) fitted on selected features

In [ ]:
# Step 1: SHAP-selected features
top_feats = mean_shap.sort_values(ascending=False).head(5).index.tolist()
print(f"Top SHAP features: {top_feats}")

# Step 2: Map back to original column names, build X and y
feat_orig = [k for k, v in SHAP_RENAME.items() if v in top_feats]
X_tree = df_shap[feat_orig].rename(columns=SHAP_RENAME)

# Binary target: High-skill = 1, Low/Moderate = 0
y_tree = (df_shap['Cluster'] == 2).astype(int)

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_tree, y_tree)
acc = (dt.predict(X_tree) == y_tree).mean()
print(f"Surrogate tree accuracy: {acc:.3f}  (n={len(X_tree)})")

### Figure: Surrogate Decision Tree (Fig. 7 in paper)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(dt,
          feature_names=top_feats,
          class_names=['Low/Moderate', 'High'],
          filled=True, rounded=True,
          fontsize=10, ax=ax,
          impurity=True, proportion=False)
ax.set_title(
    'Surrogate Decision Tree (depth=3, accuracy=93.3%)\n'
    'Fitted on SHAP-selected features',
    fontsize=12, pad=15)
plt.tight_layout()
plt.savefig(f'{OUT}/fig5_rules_new.png', bbox_inches='tight',
            dpi=180, facecolor='white')
plt.show()

## 7. Supplementary: Cluster Profile Heatmap

In [ ]:
heatmap_cols   = ['All_Skill','Avg_ProdManage','Avg_Tech','Avg_Mkting',
                  'age','edu','Ave_MktRisk', OUTCOME]
heatmap_labels = ['All_Skill','Avg_ProdManage','Avg_Tech','Avg_Mkting',
                  'Age','Education','Mkt Risk','Ch_Skill']

rows = []
for ci in range(3):
    rows.append(df_full[df_full['Cluster']==ci][heatmap_cols].mean())
heat_df = pd.DataFrame(rows,
    index=['Low-skill','Moderate-skill','High-skill'],
    columns=heatmap_labels)

fig, ax = plt.subplots(figsize=(10, 3.5))
sns.heatmap(heat_df, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, annot_kws={'size':11}, ax=ax)
ax.set_title('Cluster Profile: Mean Variable Values by Farmer Segment',
             fontsize=12, pad=10)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(f'{OUT}/fig_cluster_profile.png', bbox_inches='tight',
            dpi=180, facecolor='white')
plt.show()

---
## Files Generated

| File | Description |
|---|---|
| `figures/fig_elbow_silhouette.png` | 4-panel cluster validity |
| `figures/fig2_clusters_new.png`    | PCA scatter k=3 |
| `figures/fig3_importance_new.png`  | Global SHAP bar chart |
| `figures/fig4_shap_new.png`        | Global SHAP beeswarm |
| `figures/fig4b_cluster_shap_new.png` | Cluster SHAP 3-panel |
| `figures/fig5_rules_new.png`       | Surrogate decision tree |
| `figures/fig_cluster_profile.png`  | Cluster heatmap (supplementary) |